# Sector Rotation System -- Setup & Backtest

This notebook runs on **Google Colab** (free tier). It:

1. Clones the repo and installs dependencies
2. Populates the database with 2 years of historical data
3. Runs the full pipeline (regime -> screener -> optimizer -> NLP)
4. Executes a walk-forward backtest and displays results
5. Shows the current Executive Summary with dollar allocations
6. **Reviews portfolio** -- confirm/edit before downloading

**Runtime:** ~5-8 minutes on Colab free tier.

## 1. Clone Repo & Install Dependencies

In [1]:
# --- Clone private repo using Colab Secrets ---
# Store your GitHub PAT in Colab Secrets (key icon in sidebar) as GITHUB_TOKEN.
# The token needs 'repo' scope for private repository access.

import os
import importlib
from pathlib import Path

REPO_DIR = '/content/sector-rotation-system'

if not os.path.isdir(REPO_DIR):
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    !git clone https://{token}@github.com/bigbathtub101/sector-rotation-system.git {REPO_DIR}
else:
    print(f'Repo already cloned at {REPO_DIR}, pulling latest...')
    !cd {REPO_DIR} && git pull
    # -- MODULE CACHE FIX ------------------------------------------
    # After git pull, Python's module cache still holds the OLD code.
    # Force-reload every project module so the new code takes effect.
    import sys
    project_modules = [
        'data_feeds', 'regime_detector', 'portfolio_optimizer',
        'stock_screener', 'nlp_sentiment', 'etf_selector', 'monitor',
    ]
    for mod_name in project_modules:
        if mod_name in sys.modules:
            importlib.reload(sys.modules[mod_name])
            print(f'  ~ Reloaded {mod_name}')
    # --------------------------------------------------------------

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Install dependencies
!pip install -q -r requirements.txt
print('\nSetup complete.')

Cloning into '/content/sector-rotation-system'...
remote: Enumerating objects: 544, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (190/190), done.
remote: Total 544 (delta 109), reused 6 (delta 6), pack-reused 343 (from 1)
Receiving objects: 100% (544/544), 742.43 KiB | 7.28 MiB/s, done.
Resolving deltas: 100% (276/276), done.
Working directory: /content/sector-rotation-system
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 8.5 MB/s eta 0:00:00

Setup complete.


## 2. Set API Keys (Optional)

FRED key is optional -- the system works without it using placeholder macro data.
Set it here if you have one.

In [2]:
import os
import re

# Optional: set your FRED API key via Colab Secrets (free at https://fred.stlouisfed.org)
# Store your key in Colab Secrets (key icon in sidebar) as FRED_API_KEY.
try:
    from google.colab import userdata
    fred_key = userdata.get("FRED_API_KEY")
    if fred_key:
        os.environ["FRED_API_KEY"] = fred_key
except Exception:
    pass

# Optional: set your SEC EDGAR email via Colab Secrets.
try:
    from google.colab import userdata
    sec_email = userdata.get("SEC_EDGAR_EMAIL")
    if sec_email:
        # Aggressively strip ALL control characters (Colab Secrets injects \r\n)
        sec_email = re.sub(r"[^\x20-\x7E]", "", sec_email).strip()
        if sec_email:
            os.environ["SEC_EDGAR_EMAIL"] = sec_email
        else:
            print("WARNING: SEC_EDGAR_EMAIL contained only control characters -- ignoring.")
except Exception:
    pass

# Verify
fred_key = os.environ.get("FRED_API_KEY", "")
print(f"FRED_API_KEY: {('set (' + fred_key[:4] + '...)') if fred_key else 'NOT SET (will use dummy macro data)'}")
sec_email = os.environ.get("SEC_EDGAR_EMAIL", "")
print(f"SEC_EDGAR_EMAIL: {('set (' + repr(sec_email[:20]) + '...)') if sec_email else 'NOT SET (will use config.yaml default)'}")
if sec_email:
    print(f"  repr check: {repr(sec_email)}")


FRED_API_KEY: set (a5c3...)
SEC_EDGAR_EMAIL: set ('gabop98@gmail.com'...)
  repr check: 'gabop98@gmail.com'


## 3. Load Config & Backfill Historical Data

Downloads ~2 years of daily prices for all ETFs and watchlist stocks via yfinance,
macro data from FRED (if key is set), and recent SEC filings.

In [3]:
import yaml
import sqlite3
import inspect
import importlib
import pandas as pd
from pathlib import Path

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# ----- CRITICAL: Define a single DB path used by EVERYTHING -----
DB_PATH = Path(os.getcwd()) / 'rotation_system.db'

# --- Helper: patch a module's DB_PATH everywhere it matters ---
def patch_db_path(mod, new_path):
    """Overwrite a module's DB_PATH global AND fix any function
    defaults that captured the old value at import time."""
    old_path = getattr(mod, 'DB_PATH', None)
    mod.DB_PATH = new_path
    for name, obj in inspect.getmembers(mod, inspect.isfunction):
        if obj.__defaults__:
            new_defaults = []
            changed = False
            for d in obj.__defaults__:
                if isinstance(d, Path) and old_path and d == old_path:
                    new_defaults.append(new_path)
                    changed = True
                else:
                    new_defaults.append(d)
            if changed:
                obj.__defaults__ = tuple(new_defaults)
    print(f'  {mod.__name__}.DB_PATH -> {mod.DB_PATH}')

print("Config loaded. Portfolio total: ${:,.0f}".format(config['portfolio']['total_value']))
print(f"  Taxable: ${config['portfolio']['accounts']['taxable']['value']:,.0f}")
print(f"  Roth IRA: ${config['portfolio']['accounts']['roth_ira']['value']:,.0f}")
print(f"  Sector ETFs: {len(config['tickers']['sector_etfs'])}")
wl_keys = ['watchlist_biotech', 'watchlist_ai_software', 'watchlist_defense', 'watchlist_green_materials']
wl_count = sum(len(config['tickers'].get(k, [])) for k in wl_keys)
print(f"  Watchlist tickers: {wl_count}")
print(f"  DB path: {DB_PATH}")

Config loaded. Portfolio total: $144,000
  Taxable: $100,000
  Roth IRA: $44,000
  Sector ETFs: 11
  Watchlist tickers: 47
  DB path: /content/sector-rotation-system/rotation_system.db


In [4]:
import gc
import sqlite3
import datetime as dt

# Force cleanup of any stale DB connections from prior cell runs
gc.collect()

# Verify DB is accessible before proceeding
if DB_PATH.exists():
    try:
        _test_conn = sqlite3.connect(str(DB_PATH), timeout=30)
        _test_conn.execute("PRAGMA journal_mode=WAL")
        _test_conn.execute("SELECT 1")
        _test_conn.close()
        print(f"[OK] DB file verified: {DB_PATH}")
    except sqlite3.OperationalError as e:
        print(f"[!] DB appears locked from a prior run, removing stale file: {e}")
        DB_PATH.unlink()

import data_feeds
importlib.reload(data_feeds)  # <- ensure latest code after git pull
patch_db_path(data_feeds, DB_PATH)

# Single shared connection -- eliminates competing connections that cause DB lock
conn = data_feeds.init_database(DB_PATH)

# Thresholds for incremental ingestion
PRICE_FRESHNESS_DAYS = 3   # skip price download if DB is this fresh
MIN_FILINGS_THRESHOLD = 50  # skip filings download if DB has more than this

# --- Patch for NameError: name 'lookup_cik' is not defined ---
if not hasattr(data_feeds, 'lookup_cik'):
    import logging
    logger = logging.getLogger(__name__)
    def _placeholder_lookup_cik(ticker, cfg, log_warning=True):
        if log_warning:
            logger.warning("Using placeholder lookup_cik for ticker %s.", ticker)
        return None
    data_feeds.lookup_cik = _placeholder_lookup_cik
    print("[!]  Applied temporary patch: `data_feeds.lookup_cik` placeholder added.")

try:
    # --- Incremental Prices ---
    print("\n[>>] Checking price data...")
    latest_date = None
    try:
        row = conn.execute("SELECT MAX(date) FROM prices").fetchone()
        if row and row[0]:
            latest_date = row[0]
    except Exception:
        pass

    today = dt.date.today()
    if latest_date and (today - dt.date.fromisoformat(latest_date)).days <= PRICE_FRESHNESS_DAYS:
        print(f"[OK] Prices are current (latest: {latest_date}). Skipping download.")
        prices_summary = {"skipped": True}
    else:
        if latest_date:
            price_start = (dt.date.fromisoformat(latest_date) + dt.timedelta(days=1)).isoformat()
            print(f"[>>] Fetching incremental prices from {price_start} to today...")
        else:
            price_start = None
            print("[>>] First run -- fetching full 2-year price history...")
        prices = data_feeds.fetch_prices(config, start_date=price_start)
        prices, price_warnings = data_feeds.validate_prices(prices, config)
        data_feeds.store_prices(conn, prices)
        prices_summary = {
            "rows": len(prices),
            "tickers": int(prices["ticker"].nunique()) if not prices.empty else 0,
        }
        if price_warnings:
            print(f"  [!] {len(price_warnings)} price warnings")
        print(f"[OK] Prices stored: {prices_summary}")

    # --- Macro (always refresh -- fast ~5s) ---
    print("\n[>>] Fetching macro data from FRED...")
    macro = data_feeds.fetch_macro_data(config)
    macro, macro_warnings = data_feeds.validate_macro(macro)
    data_feeds.store_macro(conn, macro)
    macro_summary = {
        "rows": len(macro),
        "series": int(macro["series_id"].nunique()) if not macro.empty else 0,
    }
    print(f"[OK] Macro stored: {macro_summary}")

    # --- SEC Filings (skip if already populated) ---
    print("\n[>>] Checking SEC filings...")
    filing_count = conn.execute("SELECT COUNT(*) FROM filings").fetchone()[0]
    if filing_count > MIN_FILINGS_THRESHOLD:
        print(f"[OK] Filings already populated ({filing_count:,} rows). Skipping download.")
        print("   To force a refresh, delete the DB file and re-run this cell.")
        filings_summary = {"skipped": True, "existing": filing_count}
    else:
        import time as _time
        max_budget = config.get("sec_edgar", {}).get("max_fetch_time_seconds", 300)
        print(f"[>>] First run -- fetching SEC filings (budget: {max_budget}s)...")
        _t0 = _time.monotonic()
        filings = data_feeds.fetch_all_filings(config)
        _elapsed = _time.monotonic() - _t0
        data_feeds.store_filings(conn, filings)
        with_text = sum(1 for f in filings if f.get("raw_text"))
        metadata_only = len(filings) - with_text
        filings_summary = {"count": len(filings), "with_text": with_text, "metadata_only": metadata_only}
        print(f"[OK] Filings stored: {len(filings):,} total in {_elapsed:.0f}s "
              f"({with_text:,} with text, {metadata_only:,} metadata-only)")
        if len(filings) == 0:
            print("  [!]  WARNING: 0 filings were fetched.")

finally:
    conn.close()

# --- Ticker Coverage Report ---
import pandas as pd
print(f"\n=== Ticker Coverage ===")
print(f"  DB file exists: {DB_PATH.exists()}")
if DB_PATH.exists():
    print(f"  DB file size: {DB_PATH.stat().st_size:,} bytes")
    conn2 = sqlite3.connect(str(DB_PATH))
    try:
        for table in ['prices', 'macro_data', 'filings', 'signals']:
            try:
                count = pd.read_sql(f'SELECT COUNT(*) AS n FROM {table}', conn2).iloc[0]['n']
                print(f"  {table}: {count:,} rows")
            except Exception as e:
                print(f"  {table}: ERROR - {e}")
        loaded = pd.read_sql('SELECT DISTINCT ticker FROM prices', conn2)['ticker'].tolist()
        all_expected = set()
        for key in config['tickers']:
            if isinstance(config['tickers'][key], list):
                all_expected.update(config['tickers'][key])
        missing = all_expected - set(loaded)
        print(f"\n  Tickers loaded: {len(loaded)} / {len(all_expected)} expected")
        if missing:
            print(f"  Missing tickers ({len(missing)}): {sorted(missing)}")
            print(f"  NOTE: OTC ADRs (BAESY, RNMBY, SAABF) and VIX indices")
            print(f"  often fail on yfinance. This is expected.")
    finally:
        conn2.close()
else:
    print("  ERROR: DB file was NOT created! Check ingestion logs above.")

  data_feeds.DB_PATH -> /content/sector-rotation-system/rotation_system.db

[>>] Checking price data...
[>>] First run -- fetching full 2-year price history...


[*********************100%***********************]  151 of 151 completed


  [!] 1 price warnings
[OK] Prices stored: {'rows': 75500, 'tickers': 151}

[>>] Fetching macro data from FRED...
[OK] Macro stored: {'rows': 611, 'series': 6}

[>>] Checking SEC filings...
[>>] First run -- fetching SEC filings (budget: 300s)...


[OK] Filings stored: 226 total in 96s (226 with text, 0 metadata-only)

=== Ticker Coverage ===
  DB file exists: True
  DB file size: 25,993,216 bytes
  prices: 75,500 rows
  macro_data: 611 rows
  filings: 224 rows
  signals: 0 rows

  Tickers loaded: 151 / 151 expected


## 4. Detect Current Regime

In [5]:
import regime_detector
importlib.reload(regime_detector)  # <- ensure latest code after git pull
patch_db_path(regime_detector, DB_PATH)

regime_result = regime_detector.run_regime_detection(config)

print("=" * 60)
print("CURRENT REGIME")
print("=" * 60)
print(f"  Regime:              {regime_result.get('dominant_regime', 'N/A')}")
print(f"  Wedge Volume %:      {regime_result.get('wedge_volume_percentile', 'N/A')}")
print(f"  Fast Shock Risk:     {regime_result.get('fast_shock_risk', 'N/A')}")
print(f"  VIX/RV Ratio:        {regime_result.get('vix_rv_ratio', 'N/A')}")
print(f"  Confirmed:           {regime_result.get('regime_confirmed', 'N/A')}")
print(f"  Consecutive Days:    {regime_result.get('consecutive_days_in_regime', 'N/A')}")

  regime_detector.DB_PATH -> /content/sector-rotation-system/rotation_system.db
CURRENT REGIME
  Regime:              offense
  Wedge Volume %:      85.38
  Fast Shock Risk:     high
  VIX/RV Ratio:        1.6849
  Confirmed:           True
  Consecutive Days:    37


## 5. Stock Screener -- Top Picks

Scores all watchlist stocks and identifies the **top 5 picks** for the Roth IRA
satellite sleeve (20% of Roth = ~$8,800). This must run before the CVaR optimizer
so the optimizer can incorporate these picks into the final allocation.

In [6]:
import stock_screener
importlib.reload(stock_screener)  # <- ensure latest code
patch_db_path(stock_screener, DB_PATH)

regime = regime_result.get('dominant_regime', 'offense')
screener_result = stock_screener.run_stock_screener(cfg=config, regime=regime)

print("=" * 60)
print("THEMATIC WATCHLIST SCORES")
print("=" * 60)

watchlists = screener_result.get('watchlist_scores', {})
for name, data in watchlists.items():
    if data:
        print(f"\n--- {name.upper()} ---")
        if isinstance(data, list):
            df = pd.DataFrame(data)
        elif hasattr(data, 'to_string'):
            df = data
        else:
            print(f"  (unexpected format: {type(data).__name__})")
            continue
        if not df.empty:
            show_cols = [c for c in ['ticker', 'composite_score', 'momentum_score',
                                      'quality_score', 'value_score', 'size_score',
                                      'valuation_label', 'signal'] if c in df.columns]
            if show_cols:
                print(df[show_cols].head(10).to_string(index=False))
            else:
                print(df.head(10).to_string(index=False))
        else:
            print("  (empty)")

# Show ENTRY/EXIT signals
signals_out = screener_result.get('signals', {})
if signals_out:
    print(f"\n{'='*60}")
    print("ENTRY / EXIT SIGNALS")
    print(f"{'='*60}")
    for sig_type in ['entry', 'exit']:
        sigs = signals_out.get(sig_type, [])
        if not sigs:
            print(f"\n--- {sig_type.upper()}: none ---")
            continue
        print(f"\n--- {sig_type.upper()} ---")
        watchlist_sigs = [s for s in sigs if s.get('source', 'watchlist') != 'sector_screen']
        sector_sigs = [s for s in sigs if s.get('source') == 'sector_screen']
        if watchlist_sigs:
            print("  [Watchlist]")
            for s in watchlist_sigs[:10]:
                print(f"    {s.get('ticker', '?')} ({s.get('watchlist', '')}): {s.get('reason', '')}")
        if sector_sigs:
            print("  [Sector Screen]")
            for s in sector_sigs[:10]:
                print(f"    {s.get('ticker', '?')} ({s.get('watchlist', '')}): {s.get('reason', '')}")

# Sector Top Picks
print(f"\n{'='*60}")
print("SECTOR TOP PICKS (best stocks from overweight sectors)")
print(f"{'='*60}")
stp_data = screener_result.get('sector_top_picks', [])
if stp_data:
    stp_df = pd.DataFrame(stp_data)
    show_cols = [c for c in ['ticker', 'etf', 'composite_score', 'momentum_rank',
                               'quality_score', 'value_score', 'valuation_label'] if c in stp_df.columns]
    print(stp_df[show_cols].to_string(index=False))
else:
    print("  (no sector top picks -- no overweight sectors identified)")

# Confirm screener_output.json was written for the optimizer to read
from pathlib import Path as _Path
_screener_json = _Path(os.getcwd()) / 'screener_output.json'
if _screener_json.exists():
    print(f"\n[OK] screener_output.json written ({_screener_json.stat().st_size:,} bytes)")
    print("     The CVaR optimizer (next section) will read this to inject satellite picks.")
else:
    print("\n[!] screener_output.json NOT found -- satellite picks will not be injected.")


  stock_screener.DB_PATH -> /content/sector-rotation-system/rotation_system.db


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['HES', 'PXD']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}


THEMATIC WATCHLIST SCORES

--- BIOTECH ---
ticker  composite_score  quality_score valuation_label
  TERN           3.2937         0.0579 FUNDAMENTAL_BUY
  GPCR           0.7658         0.0415 FUNDAMENTAL_BUY
  HALO           0.6615         0.8833 FUNDAMENTAL_BUY
  IONS           0.6371         0.0528 FUNDAMENTAL_BUY
  EXEL           0.6048         0.8657 FUNDAMENTAL_BUY
  NBIX           0.5419         0.6425 FUNDAMENTAL_BUY
  ALNY           0.4881         0.7198 FUNDAMENTAL_BUY
  BMRN           0.4599         0.6349 FUNDAMENTAL_BUY
  MRNA           0.1788         0.0073 FUNDAMENTAL_BUY
  VKTX           0.1313         0.0279 FUNDAMENTAL_BUY

--- AI_SOFTWARE ---
ticker  composite_score  quality_score  value_score valuation_label
  FSLY           0.4864         0.3835       0.0084           AVOID
  DOCN           0.4301         0.6785       0.1561   MOMENTUM_ONLY
  PLTR           0.3990         0.6273       0.0007     EXCEEDS_CAP
  OKTA           0.3650         0.6036       0.4958 FUNDAME

## 6. Run CVaR Optimization

Runs the full CVaR optimizer with Ledoit-Wolf shrinkage and
Longin-Solnik tail correlations. Uses the **ETF auto-selector** to pick
best-in-class funds (e.g. FTEC over XLK, SGOV over BIL).

**Satellite stocks:** The optimizer reads `screener_output.json` (from Section 5)
and injects the top 5 screener picks into 20% of the Roth IRA (~$8,800).
Dollar amounts are split between Taxable ($100K) and Roth IRA ($44K) using tax-location rules.

In [7]:
import portfolio_optimizer
importlib.reload(portfolio_optimizer)  # <- ensure latest code after git pull
patch_db_path(portfolio_optimizer, DB_PATH)

# Also reload the ETF selector (imported by portfolio_optimizer)
try:
    import etf_selector
    importlib.reload(etf_selector)
    # Re-patch the reference inside portfolio_optimizer
    portfolio_optimizer._get_etf_selections = etf_selector.get_selected_tickers
    portfolio_optimizer._get_selector_asset_class_map = etf_selector.get_ticker_asset_class_map
    portfolio_optimizer._ETF_SELECTOR_AVAILABLE = True
    print("  [OK] ETF auto-selector loaded and active")
except ImportError:
    print("  [!] ETF selector not available -- using legacy SPDR tickers")

opt_result = portfolio_optimizer.run_portfolio_optimization(cfg=config, regime=regime)

total = config['portfolio']['total_value']
taxable_val = config['portfolio']['accounts']['taxable']['value']
roth_val = config['portfolio']['accounts']['roth_ira']['value']

print("=" * 80)
print(f"CVaR-OPTIMIZED ALLOCATION (Regime: {regime.upper()})")
print(f"Portfolio: ${total:,.0f} = ${taxable_val:,.0f} Taxable + ${roth_val:,.0f} Roth IRA")
print("=" * 80)

positions = opt_result.get('positions', {})
if positions:
    # Separate ETF and stock positions for display
    etf_positions = {t: i for t, i in positions.items()
                     if i.get('reason', '').find('satellite stock') == -1}
    stock_positions = {t: i for t, i in positions.items()
                       if i.get('reason', '').find('satellite stock') != -1}

    print(f"\n{'Ticker':<8} {'Type':<6} {'Weight':>7} {'Total $':>10} {'Taxable $':>10} {'Roth $':>10}  Account")
    print("-" * 80)
    for ticker, info in sorted(etf_positions.items(), key=lambda x: -x[1].get('pct', 0)):
        pct = info.get('pct', 0)
        total_d = info.get('total_dollars', pct / 100.0 * total)
        tax_d = info.get('taxable_dollars', 0)
        roth_d = info.get('roth_dollars', 0)
        acct = info.get('account', 'N/A')
        print(f"{ticker:<8} {'ETF':<6} {pct:>6.1f}% {total_d:>10,.0f} {tax_d:>10,.0f} {roth_d:>10,.0f}  {acct}")

    if stock_positions:
        print(f"\n  -- Roth Satellite Stocks (20% of Roth from Screener Top 5) --")
        for ticker, info in sorted(stock_positions.items(), key=lambda x: -x[1].get('pct', 0)):
            pct = info.get('pct', 0)
            total_d = info.get('total_dollars', pct / 100.0 * total)
            tax_d = info.get('taxable_dollars', 0)
            roth_d = info.get('roth_dollars', 0)
            acct = info.get('account', 'N/A')
            print(f"{ticker:<8} {'STOCK':<6} {pct:>6.1f}% {total_d:>10,.0f} {tax_d:>10,.0f} {roth_d:>10,.0f}  {acct}")
    else:
        print("\n  [!] No satellite stocks injected -- check screener_output.json exists")

    # Summary by account
    tax_total = sum(info.get('taxable_dollars', 0) for info in positions.values())
    roth_total = sum(info.get('roth_dollars', 0) for info in positions.values())
    print("-" * 80)
    print(f"{'TOTAL':<8} {'':6} {'100%':>7} {total:>10,.0f} {tax_total:>10,.0f} {roth_total:>10,.0f}")
    print(f"\nAccount utilization: Taxable {tax_total/taxable_val:.0%} | Roth {roth_total/roth_val:.0%}")
    if stock_positions:
        sat_total = sum(i.get('roth_dollars', 0) for i in stock_positions.values())
        print(f"Satellite stocks: ${sat_total:,.0f} = {sat_total/roth_val:.0%} of Roth ({len(stock_positions)} picks)")

    # -- Verify ETF auto-selector is working --
    tickers_in_output = set(positions.keys())
    fidelity_check = tickers_in_output & {'FTEC', 'FHLC', 'FNCL', 'FIDU', 'FDIS', 'FSTA', 'FENY', 'FMAT', 'FUTY', 'FREL', 'FCOM'}
    franklin_check = tickers_in_output & {'FLIN', 'FLBR', 'FLJP', 'FLCH', 'FLKR', 'FLTW'}
    sgov_check = 'SGOV' in tickers_in_output
    spdr_check = tickers_in_output & {'XLK', 'XLV', 'XLF', 'XLI', 'XLP', 'XLU', 'XLC', 'XLY', 'XLE', 'XLB', 'XLRE'}
    print(f"\n-- ETF Auto-Selector Status --")
    if fidelity_check or franklin_check or sgov_check:
        print(f"  [OK] Active -- Fidelity: {sorted(fidelity_check)}, Franklin: {sorted(franklin_check)}, Cash: {'SGOV' if sgov_check else 'N/A'}")
    if spdr_check:
        print(f"  [!]  SPDR tickers still present (not substituted): {sorted(spdr_check)}")
        print(f"      If all tickers are SPDR, the selector may not have run.")
        print(f"      Try: Runtime -> Restart Runtime, then re-run all cells.")
else:
    print("No positions returned. Check optimizer logs above.")

  portfolio_optimizer.DB_PATH -> /content/sector-rotation-system/rotation_system.db
  [OK] ETF auto-selector loaded and active
CVaR-OPTIMIZED ALLOCATION (Regime: OFFENSE)
Portfolio: $144,000 = $100,000 Taxable + $44,000 Roth IRA

Ticker   Type    Weight    Total $  Taxable $     Roth $  Account
--------------------------------------------------------------------------------
FHLC     ETF      17.2%     24,747     24,747          0  taxable
VGK      ETF      15.0%     21,600     21,600          0  taxable
FCOM     ETF      13.8%     19,872     19,872          0  taxable
FIDU     ETF      12.8%     18,427     18,427          0  taxable
FTEC     ETF       9.0%     13,010     13,010          0  taxable
FDIS     ETF       6.9%      9,968          0      9,968  roth_ira
FUTY     ETF       5.9%      8,548          0      8,548  roth_ira
VWO      ETF       4.1%      5,892          0      5,892  roth_ira
FLBR     ETF       4.0%      5,756          0      5,756  roth_ira
FLIN     ETF       4.0%  

## 7. Walk-Forward Backtest

Simulates the system over the historical period, rebalancing monthly.
At each rebalance date, the CVaR optimizer runs with the regime detected
at that point to produce actual per-ETF weights. Returns are computed
using those multi-asset weights, not a simple SPY scaling factor.

In [8]:
import numpy as np
import json as json_module

import portfolio_optimizer
importlib.reload(portfolio_optimizer)  # <- ensure latest code
patch_db_path(portfolio_optimizer, DB_PATH)

total = config['portfolio']['total_value']

conn = sqlite3.connect(str(DB_PATH))

# --- Optimization universe (mirrors portfolio_optimizer) ---
opt_universe = (
    config['tickers']['sector_etfs']
    + config['tickers']['geographic_etfs']
    + ['BIL']
)
opt_universe = list(dict.fromkeys(opt_universe))

# Load ETF prices for the optimization universe
placeholders = ','.join(['?'] * len(opt_universe))
etf_prices = pd.read_sql(
    f"SELECT date, ticker, adj_close FROM prices WHERE ticker IN ({placeholders}) ORDER BY date",
    conn, params=opt_universe, parse_dates=['date']
)

# Load SPY as benchmark
spy = pd.read_sql(
    "SELECT date, adj_close FROM prices WHERE ticker='SPY' ORDER BY date",
    conn, parse_dates=['date']
).set_index('date')
spy_returns = spy['adj_close'].pct_change().dropna()

# Load regime signals
signals_df = pd.read_sql(
    "SELECT date, signal_data FROM signals WHERE signal_type='regime_state' ORDER BY date",
    conn, parse_dates=['date']
)
if len(signals_df) > 0:
    signals_df['regime'] = signals_df['signal_data'].apply(
        lambda x: json_module.loads(x).get('dominant_regime', 'offense')
        if isinstance(x, str) else 'offense'
    )
    regime_series = signals_df.set_index('date')['regime']
else:
    regime_series = pd.Series(dtype=str, name='regime')

# Build daily return matrix for ETFs
prices_pivot = etf_prices.pivot(index='date', columns='ticker', values='adj_close')
etf_daily_rets = prices_pivot.pct_change()

# Align to common trading dates with SPY
common_idx = spy_returns.index.intersection(etf_daily_rets.dropna(how='all').index)
spy_returns = spy_returns.loc[common_idx]
etf_daily_rets = etf_daily_rets.loc[common_idx]

print(f"SPY returns: {len(spy_returns)} days")
print(f"ETF universe: {len(opt_universe)} tickers")
print(f"Regime signals: {len(regime_series)} rows")

# --- Identify rebalance dates (first trading day of each month) ---
monthly_first = {}
for date in etf_daily_rets.index:
    key = (date.year, date.month)
    if key not in monthly_first:
        monthly_first[key] = date
rebalance_dates = sorted(monthly_first.values())
print(f"Rebalance dates: {len(rebalance_dates)} months")

# --- Run optimizer at each rebalance date ---
print("Running optimizer for each rebalance date (this may take ~1 min)...")
weights_schedule = {}  # date -> {ticker: weight}
for rb_date in rebalance_dates:
    past = regime_series[regime_series.index <= rb_date]
    regime_at = past.iloc[-1] if len(past) > 0 else 'offense'
    try:
        opt_res = portfolio_optimizer.run_portfolio_optimization(
            conn=conn, cfg=config, regime=regime_at
        )
        positions = opt_res.get('positions', {})
        if positions:
            raw_w = {t: info.get('pct', 0) / 100.0 for t, info in positions.items()}
            available = {t: w for t, w in raw_w.items() if t in etf_daily_rets.columns}
            total_w = sum(available.values())
            if total_w > 0:
                weights_schedule[rb_date] = {t: w / total_w for t, w in available.items()}
    except Exception as e:
        print(f"  [!] Optimizer failed for {rb_date} (regime={regime_at}): {e}")

print(f"Optimizer ran for {len(weights_schedule)}/{len(rebalance_dates)} rebalance dates")

# --- Compute daily portfolio returns using walk-forward weights ---
strategy_rets = []
current_weights = {}
for date in etf_daily_rets.index:
    if date in weights_schedule:
        current_weights = weights_schedule[date]
    if current_weights:
        row = etf_daily_rets.loc[date]
        port_ret = sum(
            w * (row[t] if pd.notna(row.get(t, float('nan'))) else 0.0)
            for t, w in current_weights.items()
        )
    else:
        port_ret = 0.0
    strategy_rets.append(port_ret)

strategy_returns = pd.Series(strategy_rets, index=etf_daily_rets.index, name='strategy_ret')

conn.close()

print(f"Strategy returns computed: {len(strategy_returns)} days")

if len(spy_returns) > 0 and len(strategy_returns) > 0:
    merged = spy_returns.to_frame('spy_ret').join(strategy_returns, how='inner')

    if len(regime_series) > 0:
        merged['regime'] = regime_series.reindex(merged.index, method='ffill').fillna('offense')
    else:
        merged['regime'] = 'offense'

    merged['spy_cum'] = (1 + merged['spy_ret']).cumprod()
    merged['strategy_cum'] = (1 + merged['strategy_ret']).cumprod()

    days = len(merged)
    ann_factor = 252 / days if days > 0 else 1

    spy_total = merged['spy_cum'].iloc[-1] - 1
    strat_total = merged['strategy_cum'].iloc[-1] - 1

    spy_ann = (1 + spy_total) ** ann_factor - 1
    strat_ann = (1 + strat_total) ** ann_factor - 1

    spy_vol = merged['spy_ret'].std() * np.sqrt(252)
    strat_vol = merged['strategy_ret'].std() * np.sqrt(252)

    spy_sharpe = spy_ann / spy_vol if spy_vol > 0 else 0
    strat_sharpe = strat_ann / strat_vol if strat_vol > 0 else 0

    def max_dd(cum_series):
        peak = cum_series.cummax()
        dd = (cum_series - peak) / peak
        return dd.min()

    spy_mdd = max_dd(merged['spy_cum'])
    strat_mdd = max_dd(merged['strategy_cum'])

    mclean_decay = config['factor_model']['mclean_pontiff_decay']
    alpha_raw = strat_ann - spy_ann
    alpha_adj = alpha_raw * mclean_decay
    mp_label = config['backtest']['mclean_pontiff_label']

    dollar_gain_spy = spy_total * total
    dollar_gain_strat = strat_total * total

    print("=" * 70)
    print("WALK-FORWARD BACKTEST RESULTS")
    print("=" * 70)
    period_start = merged.index[0].strftime('%Y-%m-%d')
    period_end = merged.index[-1].strftime('%Y-%m-%d')
    print(f"Period: {period_start} to {period_end} ({days} trading days)")
    print()
    print(f"{'Metric':<30} {'SPY (B&H)':>15} {'Strategy':>15}")
    print("-" * 60)
    print(f"{'Total Return':<30} {spy_total:>14.1%} {strat_total:>14.1%}")
    print(f"{'Annualized Return':<30} {spy_ann:>14.1%} {strat_ann:>14.1%}")
    print(f"{'Annualized Volatility':<30} {spy_vol:>14.1%} {strat_vol:>14.1%}")
    print(f"{'Sharpe Ratio':<30} {spy_sharpe:>14.2f} {strat_sharpe:>14.2f}")
    print(f"{'Max Drawdown':<30} {spy_mdd:>14.1%} {strat_mdd:>14.1%}")
    print(f"{'Dollar Gain ($144K port.)':<30} {dollar_gain_spy:>13,.0f} {dollar_gain_strat:>13,.0f}")
    print()
    print(f"Raw Alpha vs SPY:      {alpha_raw:>+.2%}")
    print(f"Adjusted Alpha (x{mclean_decay}): {alpha_adj:>+.2%}")
    print(f"  {mp_label}")

    print(f"\nRegime distribution:")
    regime_counts = merged['regime'].value_counts()
    for r, ct in regime_counts.items():
        print(f"  {r}: {ct} days ({ct/days:.0%})")
else:
    print("Not enough data for backtest. Run data ingestion first.")
    print(f"  Debug: SPY returns = {len(spy_returns)}, strategy returns = {len(strategy_returns)}")


  portfolio_optimizer.DB_PATH -> /content/sector-rotation-system/rotation_system.db
SPY returns: 499 days
ETF universe: 21 tickers
Regime signals: 247 rows
Rebalance dates: 25 months
Running optimizer for each rebalance date (this may take ~1 min)...
Optimizer ran for 25/25 rebalance dates
Strategy returns computed: 499 days
WALK-FORWARD BACKTEST RESULTS
Period: 2024-03-05 to 2026-03-02 (499 trading days)

Metric                               SPY (B&H)        Strategy
------------------------------------------------------------
Total Return                            37.3%          43.1%
Annualized Return                       17.4%          19.8%
Annualized Volatility                   16.4%          14.6%
Sharpe Ratio                             1.06           1.36
Max Drawdown                           -18.8%         -14.3%
Dollar Gain ($144K port.)             53,701        62,097

Raw Alpha vs SPY:      +2.49%
Adjusted Alpha (x0.74): +1.84%
  (in-sample, apply 26% McLean-Pontiff d

## 8. Run Full Monitor (Executive Summary)

In [9]:
# Run the full monitor pipeline and display the executive summary.
# --no-deliver skips Telegram/email/Sheets delivery (not configured in Colab).
# --db explicitly passes our DB path to avoid any path mismatch.
!python monitor.py --no-deliver --db "{DB_PATH}" 2>&1 | head -120

2026-03-03 03:40:52,230 [INFO] monitor: Custom DB path: /content/sector-rotation-system/rotation_system.db — propagating to all modules
2026-03-03 03:40:52,330 [INFO] monitor: Configuration loaded from /content/sector-rotation-system/config.yaml
2026-03-03 03:40:52,343 [INFO] monitor: ════════════════════════════════════════════════════════════
2026-03-03 03:40:52,343 [INFO] monitor:   MONITOR RUN: run_20260303_034052
2026-03-03 03:40:52,344 [INFO] monitor:   Date: 2026-03-03   Mode: LIVE
2026-03-03 03:40:52,344 [INFO] monitor: ════════════════════════════════════════════════════════════
2026-03-03 03:40:52,344 [INFO] monitor: Step 1: Data refresh...
2026-03-03 03:40:52,345 [INFO] data_feeds: Database initialized at /content/sector-rotation-system/rotation_system.db
2026-03-03 03:40:52,345 [INFO] data_feeds: ============================================================
2026-03-03 03:40:52,345 [INFO] data_feeds: STEP 1: Fetching price data
2026-03-03 03:40:52,345 [INFO] data_feeds: =====

## 8.5. Review & Confirm Portfolio

**Before downloading**, review the portfolio below. You can:
- Check that the ETF auto-selector picked the right funds
- Verify account placement (Taxable vs Roth) makes sense
- **Edit weights** by modifying the `overrides` dict in the next cell

If everything looks good, run the cell as-is (no changes needed) and proceed to Section 9.

**Note:** Satellite stock picks were injected in Section 6 using the screener results from Section 5.

In [10]:
# =======================================================================
#  PORTFOLIO REVIEW -- Edit overrides below to adjust before downloading
# =======================================================================
#
# The optimizer produced `opt_result` in Section 6 above.
# This cell lets you inspect and optionally modify it.
#
# To override a weight, add entries to the `overrides` dict below.
# Example: overrides = {'FTEC': 12.0, 'FUTY': 0.0, 'SGOV': 5.0}
# Weights are in PERCENT and will be re-normalized to 100%.
# Leave empty ({}) to accept the optimizer's output as-is.

overrides = {}   # <- Edit here. Example: {'FTEC': 15.0, 'FUTY': 0.0}

# =======================================================================

import json

positions = opt_result.get('positions', {})
total = config['portfolio']['total_value']
taxable_val = config['portfolio']['accounts']['taxable']['value']
roth_val = config['portfolio']['accounts']['roth_ira']['value']

# Build current weights from optimizer output
current_weights = {t: info['pct'] for t, info in positions.items()}

# Apply overrides
if overrides:
    print("[>>] Applying manual overrides...")
    for ticker, new_pct in overrides.items():
        old_pct = current_weights.get(ticker, 0)
        current_weights[ticker] = new_pct
        print(f"  {ticker}: {old_pct:.1f}% -> {new_pct:.1f}%")

    # Remove zero-weight positions
    current_weights = {t: w for t, w in current_weights.items() if w > 0}

    # Re-normalize to 100%
    total_pct = sum(current_weights.values())
    if total_pct > 0 and abs(total_pct - 100.0) > 0.1:
        scale = 100.0 / total_pct
        current_weights = {t: round(w * scale, 2) for t, w in current_weights.items()}
        print(f"  Re-normalized from {total_pct:.1f}% to 100.0%")

    # Re-run dollar allocation with updated weights
    weight_fracs = {t: w / 100.0 for t, w in current_weights.items()}
    importlib.reload(portfolio_optimizer)
    positions = portfolio_optimizer.allocate_dollars(weight_fracs, config)
    opt_result['positions'] = positions
    print("  [OK] Dollar allocation recalculated with overrides.\n")
else:
    print("No overrides -- using optimizer output as-is.\n")

# -- Display final portfolio for review --
print("=" * 100)
print(f"  FINAL PORTFOLIO -- {regime.upper()} REGIME")
print(f"  ${total:,.0f} = ${taxable_val:,.0f} Taxable + ${roth_val:,.0f} Roth IRA")
print("=" * 100)

# Separate ETFs and individual stocks
etf_pos = {t: info for t, info in positions.items()
           if info.get('reason', '').find('satellite stock') == -1}
stock_pos = {t: info for t, info in positions.items()
             if info.get('reason', '').find('satellite stock') != -1}

# --- ETF Core positions ---
print(f"\n-- ETF Core Positions --")
print(f"{'#':<3} {'Ticker':<8} {'Type':<6} {'Weight':>7} {'Total $':>10} {'Taxable $':>10} {'Roth $':>10}  {'Account':<10} Reason")
print("-" * 110)

sorted_etfs = sorted(etf_pos.items(), key=lambda x: -x[1].get('pct', 0))
tax_total = 0
roth_total_ = 0
for i, (ticker, info) in enumerate(sorted_etfs, 1):
    pct = info.get('pct', 0)
    total_d = info.get('total_dollars', 0)
    tax_d = info.get('taxable_dollars', 0)
    roth_d = info.get('roth_dollars', 0)
    acct = info.get('account', 'N/A')
    reason = info.get('reason', '')
    tax_total += tax_d
    roth_total_ += roth_d
    print(f"{i:<3} {ticker:<8} {'ETF':<6} {pct:>6.1f}% ${total_d:>9,.0f} ${tax_d:>9,.0f} ${roth_d:>9,.0f}  {acct:<10} {reason}")

# --- Individual Stock Satellite positions ---
if stock_pos:
    print(f"\n-- Roth IRA Satellite Stocks (Top 5 from Screener) --")
    print(f"{'#':<3} {'Ticker':<8} {'Type':<6} {'Weight':>7} {'Total $':>10} {'Taxable $':>10} {'Roth $':>10}  {'Account':<10} Reason")
    print("-" * 110)
    sorted_stocks = sorted(stock_pos.items(), key=lambda x: -x[1].get('pct', 0))
    for j, (ticker, info) in enumerate(sorted_stocks, len(sorted_etfs) + 1):
        pct = info.get('pct', 0)
        total_d = info.get('total_dollars', 0)
        tax_d = info.get('taxable_dollars', 0)
        roth_d = info.get('roth_dollars', 0)
        acct = info.get('account', 'N/A')
        reason = info.get('reason', '')
        tax_total += tax_d
        roth_total_ += roth_d
        print(f"{j:<3} {ticker:<8} {'STOCK':<6} {pct:>6.1f}% ${total_d:>9,.0f} ${tax_d:>9,.0f} ${roth_d:>9,.0f}  {acct:<10} {reason}")

print("-" * 110)
print(f"{'':3} {'TOTAL':<8} {'':6} {'100.0':>6}% ${tax_total + roth_total_:>9,.0f} ${tax_total:>9,.0f} ${roth_total_:>9,.0f}")
print(f"\n  Taxable: ${tax_total:,.0f} / ${taxable_val:,.0f} ({tax_total/taxable_val:.0%})")
print(f"  Roth:    ${roth_total_:,.0f} / ${roth_val:,.0f} ({roth_total_/roth_val:.0%})")
if stock_pos:
    stock_total = sum(info.get('total_dollars', 0) for info in stock_pos.values())
    stock_roth_pct = stock_total / roth_val * 100 if roth_val > 0 else 0
    print(f"  Satellite stocks: ${stock_total:,.0f} = {stock_roth_pct:.0f}% of Roth IRA ({len(stock_pos)} positions)")

# -- ETF selector verification --
tickers_in_output = set(positions.keys())
fidelity = tickers_in_output & {'FTEC','FHLC','FNCL','FIDU','FDIS','FSTA','FENY','FMAT','FUTY','FREL','FCOM'}
franklin = tickers_in_output & {'FLIN','FLBR','FLJP','FLCH','FLKR','FLTW'}
spdr = tickers_in_output & {'XLK','XLV','XLF','XLI','XLP','XLU','XLC','XLY','XLE','XLB','XLRE'}
print(f"\n-- ETF Auto-Selector --")
if fidelity: print(f"  Fidelity MSCI: {sorted(fidelity)}")
if franklin: print(f"  Franklin FTSE: {sorted(franklin)}")
if 'SGOV' in tickers_in_output: print(f"  Cash: SGOV [OK]")
if 'BIL' in tickers_in_output: print(f"  Cash: BIL [!] (expected SGOV)")
if spdr: print(f"  SPDR (not substituted): {sorted(spdr)}")

print("\n" + "=" * 80)
print("  [OK] If this looks correct, proceed to Section 9 to download.")
print("  [>>] To change weights, edit the `overrides` dict above and re-run this cell.")
print("=" * 80)

No overrides -- using optimizer output as-is.

  FINAL PORTFOLIO -- OFFENSE REGIME
  $144,000 = $100,000 Taxable + $44,000 Roth IRA

-- ETF Core Positions --
#   Ticker   Type    Weight    Total $  Taxable $     Roth $  Account    Reason
--------------------------------------------------------------------------------------------------------------
1   FHLC     ETF      17.2% $   24,747 $   24,747 $        0  taxable    Taxable: broad ETF, long-term hold
2   VGK      ETF      15.0% $   21,600 $   21,600 $        0  taxable    Taxable: geographic/cash/energy (foreign tax credit / low turnover)
3   FCOM     ETF      13.8% $   19,872 $   19,872 $        0  taxable    Taxable: growth sector (capital appreciation, low dividend)
4   FIDU     ETF      12.8% $   18,427 $   18,427 $        0  taxable    Taxable: growth sector (capital appreciation, low dividend)
5   FTEC     ETF       9.0% $   13,010 $   13,010 $        0  taxable    Taxable: growth sector (capital appreciation, low dividend)
6  

## 9. Download Database

Download the populated database to your local machine for use with
the Streamlit dashboard or further analysis.

In [11]:
from google.colab import files

# Save the reviewed allocation to JSON so the dashboard picks it up
import json
alloc_path = DB_PATH.parent / 'current_allocation.json'
with open(alloc_path, 'w') as f:
    json.dump(opt_result, f, indent=2, default=str)
print(f"Allocation saved to {alloc_path}")

# Download the populated database
files.download(str(DB_PATH))
print("Database downloaded. Place it in the repo root for the dashboard.")

Allocation saved to /content/sector-rotation-system/current_allocation.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Database downloaded. Place it in the repo root for the dashboard.
